In [1]:
from transformers import pipeline
import os
import json
from datetime import datetime
import gradio as gr
import logging
from sentence_transformers import SentenceTransformer, util
from gliner import GLiNER

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
os.makedirs("logs", exist_ok=True)
logging.basicConfig(filename="logs/transcribe.log", level=logging.INFO, format="%(message)s")

In [3]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en",
               model_kwargs={"cache_dir": "models"})

print('asr_freq:', asr.feature_extractor.sampling_rate)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 12941.21it/s]


asr_freq: 16000


In [4]:
intent_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                                    device="cuda",
                                    model_kwargs={"cache_dir": "models"})

intents = ["news", "ecommerce"]
intent_descriptions = ["latest news or headlines about a person or topic",
                       "shop for a product on Amazon, eBay, or Flipkart"]
intent_emb = intent_model.encode(intent_descriptions, normalize_embeddings=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s]


In [5]:
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1",
                                    map_location="cuda",
                                    cache_dir="models")

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2050.00it/s]


In [6]:
def inference_hardware():
    on_gpu = asr.device.type == "cuda"
    in_colab = "COLAB_RELEASE_TAG" in os.environ
    if on_gpu and in_colab:
        return "colab gpu"
    if on_gpu:
        return "local gpu"
    return "cpu"

def log_transcription(filepath, text, latency_ms):
    if filepath is not None:
        audio_dir = os.path.abspath("logs/audio")
        src = os.path.abspath(filepath)
        if src.startswith(audio_dir + os.sep):
            stem = "".join(c for c in text.strip() if c.isalnum() or c in " ,.'-")[:80].strip() or "audio"
            ext = os.path.splitext(src)[1]
            dest = os.path.join(audio_dir, stem + ext)
            i = 2
            while os.path.exists(dest):
                dest = os.path.join(audio_dir, f"{stem} {i}{ext}")
                i += 1
            os.replace(src, dest)
            os.rmdir(os.path.dirname(src))
            filepath = dest
    filepath_rel = None if filepath is None else os.path.relpath(filepath)
    logging.info(json.dumps({"time": datetime.now().isoformat(), "input": filepath_rel, "output": text, "latency": f'{latency_ms} ms',
        "model": asr.model.name_or_path, "hardware": inference_hardware()}))

In [7]:
def transcribe_speech(filepath):
    latency_ms = None
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        text = ""
    else:
        started = datetime.now()
        output = asr(filepath)
        latency_ms = round((datetime.now() - started).total_seconds() * 1000)
        text = output["text"]
    log_transcription(filepath, text, latency_ms)
    return text

In [8]:
def detect_intent(text):
    started = datetime.now()
    emb = intent_model.encode(text, normalize_embeddings=True)
    scores = util.cos_sim(emb, intent_emb)[0]
    probs = scores.softmax(dim=0)
    i = int(probs.argmax())
    latency_ms = round((datetime.now() - started).total_seconds() * 1000)
    return intents[i], round(float(probs[i]), 2), latency_ms

In [ ]:
INTENT_LABELS = {
    "news": ["person", "organization", "place"],
    "ecommerce": ["item", "product"],
}

def extract_query(text, intent):
    spans = list(dict.fromkeys(
        e["text"] for e in ner_model.predict_entities(text, INTENT_LABELS[intent], threshold=0.4)
    ))
    if not spans:
        return "", 0.0
    emb = intent_model.encode([text, *spans], normalize_embeddings=True)
    target = intent_emb[intents.index(intent)]
    scores = util.cos_sim(emb[0], emb[1:])[0] + 2 * util.cos_sim(target, emb[1:])[0]
    probs = scores.softmax(dim=0)
    i = int(probs.argmax())
    return spans[i], round(float(probs[i]), 2)

In [69]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    intent = detect_intent(text)[0]
    query = extract_query(text, intent)[0]
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 226.28it/s]


Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'



Batches: 100%|██████████| 1/1 [00:00<00:00, 214.88it/s]


What's up with Trump these days?
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 299.89it/s]


...

get me some addidas sneakers from flipkart
ok ----- intent: ecommerce, query: 'addidas sneakers'



Batches: 100%|██████████| 1/1 [00:00<00:00, 288.65it/s]


Trump news
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 300.02it/s]

looking for a cheap office chair
ok ----- intent: ecommerce, query: 'office chair'

ok: 15/15


In [ ]:
mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)
os.makedirs("logs/audio", exist_ok=True)
mic_transcribe.input_components[0].GRADIO_CACHE = os.path.abspath("logs/audio")

file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)

demo = gr.Blocks()

with demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True, 
            server_port=int(os.environ.get('PORT1', 7680)))

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://3f7f7f6e89322b5f8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
demo.close()